# Public Dataset Benchmark
link: https://data.mendeley.com/datasets/9vr83n7z5j/2
DOI: 10.17632/9vr83n7z5j.2

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.naive_bayes import GaussianNB
import warnings
import os
import glob
warnings.filterwarnings('ignore')

# TNB Functions (Exact Mathematics of the Article)

In [39]:
def estimate_tnb_parameters(window):
    T = len(window)
    if T < 2: return 0, 0, 1e-6
    mu = np.mean(window[1:])
    alpha = 0.0
    for _ in range(5):
        num = np.sum((window[1:] - mu) * window[:-1])
        den = np.sum(window[:-1]**2)
        alpha = num / (den + 1e-8)
        mu = np.mean(window[1:] - alpha * window[:-1])
    sigma = np.sqrt(np.mean((window[1:] - mu - alpha * window[:-1])**2))
    return mu, alpha, max(sigma, 1e-6)

def tnb_log_posterior(window, mu, alpha, sigma):
    if len(window) < 2: return -np.inf
    x_t = window[1:]
    x_t_minus_1 = window[:-1]
    log_likelihoods = -0.5 * np.log(2 * np.pi * sigma**2) - \
                      ((x_t - mu - alpha * x_t_minus_1)**2) / (2 * sigma**2)
    return np.sum(log_likelihoods)

# Sliding Window Extraction


In [40]:
# S = Safe / Smooth / Slow (NORMAL)
# R = Rash / Rough / Rapid (AGGRESSIVE)
# E = Expressway / Event (Neste caso vamos agrupar como NORMAL ou descartar. Para manter binário, vamos usar S e R).
WINDOW_SIZE = 100 # 2 segundos a 50Hz

def load_mendeley_data(root_dir):
    windows, labels, features_base = [], [], []
    
    # Busca todas as subpastas
    subfolders = [f.path for f in os.scandir(root_dir) if f.is_dir()]
    
    for folder in subfolders:
        folder_name = os.path.basename(folder)
        # Extrai a última letra do nome da pasta (ex: "Day-1R" -> "R")
        category_char = folder_name[-1].upper() 
        
        # Mapeamento do Ground Truth
        if category_char == 'R':
            label = 'AGGRESSIVE'
        elif category_char == 'S':
            label = 'NORMAL'
        else:
            continue # Ignora as pastas 'E' (Eventos diversos) para focar na métrica de segurança
            
        acc_file = os.path.join(folder, 'Accelerometer.csv')
        
        if os.path.exists(acc_file):
            df = pd.read_csv(acc_file)
            
            # Limpeza caso alguma linha tenha NaN
            df.dropna(inplace=True)
            
            # Alguns arquivos podem ter as colunas em minúsculo, forçamos para evitar erros
            if 'X' in df.columns:
                df['magnitude'] = np.sqrt(df['X']**2 + df['Y']**2 + df['Z']**2)
            elif 'x' in df.columns:
                df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
            else:
                continue
                
            for i in range(0, len(df) - WINDOW_SIZE, WINDOW_SIZE):
                window = df['magnitude'].iloc[i:i+WINDOW_SIZE].values
                windows.append(window)
                labels.append(label)
                features_base.append([np.mean(window), np.std(window)])
                
    return np.array(windows), np.array(features_base), np.array(labels)

print("Percorrendo as pastas do Mendeley e extraindo o Ground Truth Oficial...")
# Caminho da pasta "Daywise data"
X_raw, X_base, y = load_mendeley_data('../data/Daywise data')

print(f"Total de Janelas: {len(y)} | Normal (S): {sum(y=='NORMAL')} | Aggressive (R): {sum(y=='AGGRESSIVE')}")

X_raw_train, X_raw_test, X_base_train, X_base_test, y_train, y_test = train_test_split(
    X_raw, X_base, y, test_size=0.3, random_state=42, stratify=y
)

Percorrendo as pastas do Mendeley e extraindo o Ground Truth Oficial...
Total de Janelas: 9721 | Normal (S): 7289 | Aggressive (R): 2432


# Training and Benchmarking

In [41]:
gnb = GaussianNB()
gnb.fit(X_base_train, y_train)
y_pred_gnb = gnb.predict(X_base_test)

print("\n========== CLASSIC GAUSSIAN NAIVE BAYES ==========")
print(classification_report(y_test, y_pred_gnb))
acc_gnb = accuracy_score(y_test, y_pred_gnb)

classes = np.unique(y_train)
tnb_models = {}

for cls in classes:
    class_windows = X_raw_train[y_train == cls]
    params = [estimate_tnb_parameters(w) for w in class_windows]
    tnb_models[cls] = {
        'mu': np.mean([p[0] for p in params]),
        'alpha': np.mean([p[1] for p in params]),
        'sigma': np.mean([p[2] for p in params])
    }

y_pred_tnb = []
for window in X_raw_test:
    best_class = None
    max_log_post = -np.inf
    for cls in classes:
        model = tnb_models[cls]
        score = tnb_log_posterior(window, model['mu'], model['alpha'], model['sigma'])
        if score > max_log_post:
            max_log_post = score
            best_class = cls
    y_pred_tnb.append(best_class)

print("\n========== TEMPORAL NAIVE BAYES (OURS) ==========")
print(classification_report(y_test, y_pred_tnb))
acc_tnb = accuracy_score(y_test, y_pred_tnb)

print(f"\nResumo de Acurácia: TNB ({acc_tnb:.3f}) vs GaussianNB ({acc_gnb:.3f})")


========== CLASSIC GAUSSIAN NAIVE BAYES ==========
              precision    recall  f1-score   support

  AGGRESSIVE       0.86      0.65      0.75       730
      NORMAL       0.89      0.97      0.93      2187

    accuracy                           0.89      2917
   macro avg       0.88      0.81      0.84      2917
weighted avg       0.89      0.89      0.88      2917


========== TEMPORAL NAIVE BAYES (OURS) ==========
              precision    recall  f1-score   support

  AGGRESSIVE       0.80      0.95      0.87       730
      NORMAL       0.98      0.92      0.95      2187

    accuracy                           0.93      2917
   macro avg       0.89      0.93      0.91      2917
weighted avg       0.94      0.93      0.93      2917


Resumo de Acurácia: TNB (0.927) vs GaussianNB (0.888)
